In [3]:
import pandas as pd
import numpy as np
import joblib

In [5]:
model = joblib.load("premium_model.pkl")

In [7]:
df = pd.read_csv("motor_insurance_eda_ready.csv")

In [9]:
portfolio_avg_rate = df["PREMIUM RATE"].mean()

print("Portfolio Average Premium Rate:", portfolio_avg_rate)

Portfolio Average Premium Rate: 0.011994935037681013


In [11]:
# Define Prediction Function

def predict_premium(
    insurance_type,
    vehicle_type,
    vehicle_make,
    vehicle_model,
    vehicle_age,
    sum_insured
):
    
    input_df = pd.DataFrame({
        "INSURANCE TYPE": [insurance_type],
        "VEHICLE TYPE": [vehicle_type],
        "VEHICLE MAKE": [vehicle_make],
        "VEHICLE MODEL": [vehicle_model],
        "VEHICLE AGE": [vehicle_age],
        "SUM INSURED": [sum_insured]
    })
    
    pred_log = model.predict(input_df)
    
    pred_premium = np.expm1(pred_log)[0]
    
    premium_rate = pred_premium / sum_insured
    
    return pred_premium, premium_rate

In [13]:
# Example Policy Stimulation

premium, rate = predict_premium(
    insurance_type="Comprehensive",
    vehicle_type="Car",
    vehicle_make="Toyota",
    vehicle_model="Prius",
    vehicle_age=2,
    sum_insured=20000000
)

print("Predicted Premium:", premium)
print("Premium Rate:", rate)

Predicted Premium: 265212.14820781996
Premium Rate: 0.013260607410390998


In [15]:
# Compare with Portfolio

print("Portfolio Average Rate:", portfolio_avg_rate)

if rate > portfolio_avg_rate:
    print("Pricing Level: Above Portfolio Average")
else:
    print("Pricing Level: Below Portfolio Average")

Portfolio Average Rate: 0.011994935037681013
Pricing Level: Above Portfolio Average


In [17]:
# Risk Tier Classification

def risk_tier(rate):

    if rate < portfolio_avg_rate * 0.9:
        return "Low Risk"
    
    elif rate < portfolio_avg_rate * 1.1:
        return "Medium Risk"
    
    else:
        return "High Risk"

In [19]:
# Full Pricing Output

tier = risk_tier(rate)

print("----- POLICY PRICING RESULT -----")

print("Predicted Premium:", round(premium,2))
print("Premium Rate:", round(rate*100,3), "%")

print("Portfolio Avg Rate:", round(portfolio_avg_rate*100,3), "%")

print("Risk Tier:", tier)

----- POLICY PRICING RESULT -----
Predicted Premium: 265212.15
Premium Rate: 1.326 %
Portfolio Avg Rate: 1.199 %
Risk Tier: High Risk


In [21]:
# Multiple Policy Simulation

policies = [
    ("Comprehensive","Car","Toyota","Prius",2,20000000),
    ("Third Party","Bike","Honda","CBR",5,4000000),
    ("Comprehensive","Car","Honda","Civic",1,25000000)
]

for p in policies:
    
    premium, rate = predict_premium(*p)
    
    print(p)
    print("Premium:", round(premium,2))
    print("Rate:", round(rate*100,3),"%")
    print()

('Comprehensive', 'Car', 'Toyota', 'Prius', 2, 20000000)
Premium: 265212.15
Rate: 1.326 %

('Third Party', 'Bike', 'Honda', 'CBR', 5, 4000000)
Premium: 1200.0
Rate: 0.03 %

('Comprehensive', 'Car', 'Honda', 'Civic', 1, 25000000)
Premium: 332822.29
Rate: 1.331 %



In [ ]:
# Save Pricing Results

